::: {.sb-downloads}
[Download this notebook](../downloads/solution-uber-pickups.ipynb){download="solution-uber-pickups.ipynb"}  
Data: [uber_pickups_apr14.csv.gz](../downloads/data/uber_pickups_apr14.csv.gz){download="uber_pickups_apr14.csv.gz"}

Put the data files in a folder named `data` next to the notebook.
:::

::: {.sb-lesson-links}
[← Back to the project](../lessons/uber-pickups.ipynb)
:::

::: {.callout-tip}
Work on the project on your own first: you learn much more by searching than by reading a solution.
:::

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
import plotly.io as pio
import plotly.graph_objects as go
pio.templates["course"] = go.layout.Template(layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"])
pio.templates.default = "plotly_white+course"

## Part 1: explore the pickups

In [ ]:
pickups = pd.read_csv("data/uber_pickups_apr14.csv.gz")
pickups["time"] = pd.to_datetime(pickups["Date/Time"], format="%m/%d/%Y %H:%M:%S")
pickups["weekday"] = pickups["time"].dt.day_name()
pickups["hour"] = pickups["time"].dt.hour
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
print(f"{len(pickups):,} pickups from {pickups.time.min():%d %B} to {pickups.time.max():%d %B %Y}")
pickups.head()

564,516 pickups from 01 April to 30 April 2014


,Date/Time,Lat,Lon,Base,time,weekday,hour
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512,2014-04-01 00:11:00,Tuesday,0
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512,2014-04-01 00:17:00,Tuesday,0
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512,2014-04-01 00:21:00,Tuesday,0
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512,2014-04-01 00:28:00,Tuesday,0
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512,2014-04-01 00:33:00,Tuesday,0


In [ ]:
counts = pickups.pivot_table(index="weekday", columns="hour", values="Lat", aggfunc="size").loc[DAYS]
fig = px.imshow(counts, aspect="auto", color_continuous_scale="Blues",
                labels=dict(x="hour of the day", y="", color="pickups"),
                title="Number of pickups in April 2014, by day and hour")
fig.show()

**What this shows:** demand follows the working week. From Monday to Friday there is a morning peak (7–8 am) and a stronger evening peak (5–9 pm); on Friday and Saturday nights the activity stays high after midnight. Sundays are the quietest days. Hot zones will change with the hour, so we will compute them **per day and per hour**.

In [ ]:
sample = pickups.sample(20000, random_state=0)
fig = px.scatter_map(sample, lat="Lat", lon="Lon", zoom=9.5, opacity=0.3, height=550,
                     map_style="carto-positron", title="A sample of 20,000 pickups")
fig.update_traces(marker=dict(size=3))
fig.update_layout(map_center=dict(lat=sample["Lat"].mean(), lon=sample["Lon"].mean()))   # centre the map on the data
fig.show()

Most pickups are in Manhattan, with smaller groups in Brooklyn, Queens and at the two airports (LaGuardia and JFK).

## Part 2: start small, Monday 5 pm

In [ ]:
slot = pickups[(pickups.weekday == "Monday") & (pickups.hour == 17)]
print(f"{len(slot)} pickups on the Mondays of April between 5 and 6 pm")

5574 pickups on the Mondays of April between 5 and 6 pm


### K-Means

K-Means needs a number of zones $k$. Latitude and longitude are close to a flat grid at the scale of a city, so the Euclidean distance on the raw coordinates is acceptable here. We compare the inertia (elbow) and the silhouette score for $k$ from 2 to 12.

In [ ]:
coords = slot[["Lat", "Lon"]]
rows = []
for k in range(2, 13):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(coords)
    rows.append({"k": k, "inertia": km.inertia_,
                 "silhouette": silhouette_score(coords, km.labels_, sample_size=3000, random_state=0)})
scores = pd.DataFrame(rows)
fig = px.line(scores, x="k", y="silhouette", markers=True, title="Silhouette score of K-Means")
fig.show()
scores.round(3)

,k,inertia,silhouette
0,2,9.550,0.756
1,3,6.746,0.458
2,4,5.220,0.485
3,5,3.716,0.493
4,6,3.010,0.492
5,7,2.545,0.493
6,8,2.146,0.409
7,9,1.851,0.407
8,10,1.606,0.426
9,11,1.407,0.437


The silhouette is by far the highest for $k = 2$: Manhattan on one side, everything else on the other. It is a well-separated split, but a useless one for a driver! From 4 to 7 the score is stable (about 0.49), then it drops. The metric gives a range; the business question decides: we keep $k = 6$, enough zones to be useful.

In [ ]:
k_best = 6
kmeans = KMeans(n_clusters=k_best, n_init=10, random_state=0).fit(coords)
zones_km = (slot.assign(zone=kmeans.labels_).groupby("zone")
            .agg(Lat=("Lat", "mean"), Lon=("Lon", "mean"), pickups=("Lat", "size")).reset_index())
print(f"k = {k_best}")
fig = px.scatter_map(slot.assign(zone=kmeans.labels_.astype(str)), lat="Lat", lon="Lon", color="zone",
                     zoom=9.5, height=550, opacity=0.5, map_style="carto-positron",
                     title=f"K-Means, Monday 5 pm: {k_best} zones")
fig.update_traces(marker=dict(size=4))
fig.add_scattermap(lat=zones_km.Lat, lon=zones_km.Lon, mode="markers", name="zone centres",
                   marker=dict(size=14, color="#111111"))
fig.update_layout(map_center=dict(lat=slot["Lat"].mean(), lon=slot["Lon"].mean()))   # centre the map on the data
fig.show()
zones_km.sort_values("pickups", ascending=False)

k = 6


,zone,Lat,Lon,pickups
1,1,40.761117,-73.975605,2759
4,4,40.730039,-74.000179,2131
2,2,40.690451,-73.971080,368
0,0,40.797279,-73.879088,163
3,3,40.659049,-73.774256,111
5,5,40.696569,-74.186836,42


**What this shows:** K-Means divides **all** the pickups into zones. Two zones (Midtown and Lower Manhattan) hold 2,759 and 2,131 pickups, but another one is made of only 42 rides around Newark airport, and the Brooklyn and Queens zones are large and sparse: their centre is not a place where a driver will find many customers. K-Means answers "how could we split the city into territories?", not exactly "where are the hot spots?".

### DBSCAN

DBSCAN keeps only the dense places. We use the haversine distance on coordinates in radians, so `eps` is a real distance. First, the k-distance plot: for each pickup, the distance to its 25th nearest neighbour.

In [ ]:
EARTH_RADIUS_M = 6_371_000
X = np.radians(slot[["Lat", "Lon"]].to_numpy())
distances, _ = NearestNeighbors(n_neighbors=25, metric="haversine").fit(X).kneighbors(X)
k_dist = np.sort(distances[:, -1]) * EARTH_RADIUS_M
fig = px.line(x=np.arange(len(k_dist)), y=k_dist, log_y=True,
              labels=dict(x="pickups, sorted", y="distance to the 25th neighbour (m, log scale)"),
              title="k-distance plot, Monday 5 pm")
fig.show()
print("Median distance to the 25th neighbour: %.0f m" % np.median(k_dist))

Median distance to the 25th neighbour: 168 m


The curve rises slowly and then explodes: there is no single clear elbow, because the density varies a lot between Manhattan (many neighbours within 100 m) and the rest of the city (hundreds of metres). Let's see what this means for DBSCAN, with a large and a small `eps`.

In [ ]:
def dbscan_zones(slot, eps_m, min_samples):
    X = np.radians(slot[["Lat", "Lon"]].to_numpy())
    labels = DBSCAN(eps=eps_m / EARTH_RADIUS_M, min_samples=min_samples, metric="haversine",
                    algorithm="ball_tree").fit_predict(X)
    return labels

min_samples = int(0.004 * len(slot))   # 0.4% of the pickups of the slot
print("min_samples =", min_samples)
for eps_m in [400, 80]:
    labels = dbscan_zones(slot, eps_m, min_samples)
    sizes = pd.Series(labels[labels >= 0]).value_counts()
    print(f"eps = {eps_m} m: {sizes.size} clusters, {np.mean(labels == -1):.0%} noise, "
          f"biggest cluster: {sizes.iloc[0]} pickups ({sizes.iloc[0] / len(slot):.0%})")

min_samples = 22


eps = 400 m: 8 clusters, 11% noise, biggest cluster: 4731 pickups (85%)


eps = 80 m: 18 clusters, 84% noise, biggest cluster: 189 pickups (3%)


**What this shows:** with `eps` = 400 m, almost every pickup has 22 neighbours within 400 m, so DBSCAN chains them together: the biggest cluster is basically **all of Manhattan**, which is not a useful hot zone. With `eps` = 80 m, only the really dense spots become clusters (18 of them), and 84% of the pickups are "noise": here, noise does not mean an error, it means **ordinary demand**, spread over the city. The clusters are the hot spots.

In [ ]:
labels = dbscan_zones(slot, 80, min_samples)
zones_db = (slot.assign(zone=labels)[labels >= 0].groupby("zone")
            .agg(Lat=("Lat", "mean"), Lon=("Lon", "mean"), pickups=("Lat", "size")).reset_index())
fig = px.scatter_map(slot.assign(zone=labels)[labels >= 0].astype({"zone": str}), lat="Lat", lon="Lon", color="zone",
                     zoom=11, height=550, map_style="carto-positron",
                     title=f"DBSCAN (80 m, {min_samples} pickups), Monday 5 pm: {len(zones_db)} hot spots")
fig.update_traces(marker=dict(size=5))
fig.update_layout(showlegend=False)
fig.update_layout(map_center=dict(lat=slot[labels >= 0]["Lat"].mean(), lon=slot[labels >= 0]["Lon"].mean()))   # centre the map on the data
fig.show()
zones_db.sort_values("pickups", ascending=False).head(8)

,zone,Lat,Lon,pickups
8,8,40.759637,-73.973287,189
7,7,40.756427,-73.982809,94
3,3,40.755817,-73.974933,83
0,0,40.753251,-73.979856,79
6,6,40.644854,-73.782198,50
1,1,40.774036,-73.872518,45
2,2,40.727861,-74.007107,41
5,5,40.751154,-74.006385,41


**What this shows:** DBSCAN finds small, precise hot spots: the biggest ones are in Midtown (189 pickups in the busiest), but also at the two airports, JFK (50) and LaGuardia (45). These are places you could send a driver to. The sparse rides elsewhere are ignored.

### Which one answers the question?

| | K-Means | DBSCAN |
|---|---|---|
| What it returns | a split of the whole city into $k$ zones | dense spots only, the rest is "noise" |
| Number of zones | chosen by us ($k$) | found by the algorithm, from `eps` and `min_samples` |
| Shape of the zones | round, of similar size | any shape, any size |
| Good for | territories (e.g. assigning drivers to areas) | **hot spots** (where to wait for a customer) |

For "where should a driver be right now?", DBSCAN's dense spots are the better answer. K-Means is still useful for a coarser view.

## Part 3: generalize to every day and hour

The number of pickups per slot varies from a few hundred (Sunday morning) to several thousands (weekday evenings). With a fixed `min_samples`, quiet slots would get no hot spot at all, and busy slots too many. So we ask for a density **relative to the slot**: a hot spot is a place where at least 0.4% of the slot's pickups (and at least 5) are within 80 m of a core pickup, as for Monday 5 pm.

In [ ]:
def hot_zones(pickups, day, hour, eps_m=80, share=0.004):
    """Hot spots of one time slot: centre, number of pickups and share of the slot."""
    slot = pickups[(pickups.weekday == day) & (pickups.hour == hour)]
    min_samples = max(5, int(share * len(slot)))
    labels = dbscan_zones(slot, eps_m, min_samples)
    zones = (slot[labels >= 0].assign(zone=labels[labels >= 0]).groupby("zone")
             .agg(Lat=("Lat", "mean"), Lon=("Lon", "mean"), pickups=("Lat", "size")).reset_index(drop=True))
    zones["share of the slot"] = (zones.pickups / len(slot)).round(3)
    return zones.assign(weekday=day, hour=hour).sort_values("pickups", ascending=False)

hot_zones(pickups, "Monday", 17).head()

,Lat,Lon,pickups,share of the slot,weekday,hour
8,40.759637,-73.973287,189,0.034,Monday,17
7,40.756427,-73.982809,94,0.017,Monday,17
3,40.755817,-73.974933,83,0.015,Monday,17
0,40.753251,-73.979856,79,0.014,Monday,17
6,40.644854,-73.782198,50,0.009,Monday,17


In [ ]:
all_zones = pd.concat([hot_zones(pickups, day, hour) for day in DAYS for hour in range(24)], ignore_index=True)
summary = all_zones.pivot_table(index="weekday", columns="hour", values="pickups", aggfunc="size").loc[DAYS]
print(f"{len(all_zones)} hot spots over the 168 time slots")
px.imshow(summary.fillna(0), aspect="auto", color_continuous_scale="Oranges",
          labels=dict(x="hour of the day", y="", color="hot spots"), title="Number of hot spots per time slot").show()

2600 hot spots over the 168 time slots


## Part 4: the dashboard

One map per day of the week, with a slider for the hour: the size of a circle is the number of pickups of the hot spot.

In [ ]:
fig = px.scatter_map(all_zones[all_zones.weekday == "Friday"], lat="Lat", lon="Lon", size="pickups",
                     animation_frame="hour", zoom=10.5, height=600, size_max=25, map_style="carto-positron",
                     center=dict(lat=40.74, lon=-73.98), title="Hot spots on Fridays, hour by hour")
fig.show()

In [ ]:
evening = all_zones[all_zones.hour == 18]
fig = px.scatter_map(evening, lat="Lat", lon="Lon", size="pickups", color="weekday", size_max=20,
                     category_orders={"weekday": DAYS}, zoom=10.5, height=600, map_style="carto-positron",
                     center=dict(lat=40.74, lon=-73.98), title="Hot spots at 6 pm, by day of the week")
fig.show()

## Conclusion

- Demand depends strongly on the day and the hour: the hot zones must be computed **per time slot**.
- **K-Means** splits the whole city into territories: easy to explain, but its zones are not necessarily hot.
- **DBSCAN** with a haversine distance finds small, dense **hot spots**. Because the density is so different between Manhattan and the rest of the city, a large `eps` merges everything; a small `eps` (80 m) with a `min_samples` that follows the number of pickups of the slot gives useful spots.
- Next steps: validate the zones on another month (do the April hot spots still exist in May?), and try `HDBSCAN` (also in scikit-learn), designed for clusters of different densities.